# Continuous and midpoint-centered transition survival for lifetime Oλ
Use the 16 equal-weight particles from the original lifetime trajectory fit with no asymptote floor. Both models use dataset-specific baseline hazards and partially pooled dataset effects, with treatment and disease adjustments. The continuous HR is per global SD increase in $\mathcal O_\lambda$. The transition HR compares the high plateau with the low plateau. Its baseline is at sigmoid activation 0.5. Saved high-regime memberships refer to higher $\mathcal O_\lambda$.

Run in order. Each model saves to its own Drive directory. Rerunning skips matching completed fits. Changing inputs or sampling settings creates a separate run.

In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import json
import subprocess
import sys
import tempfile
import pandas as pd

drive.mount('/content/drive')
DATA = Path('/content/drive/MyDrive/Hartert/CAR-T-spectral-organization/DATA')
FIRST_STAGE = DATA / 'olambda_asymptotes_min1000_no_floor_v1'
ESTIMATES = FIRST_STAGE / 'patient_olambda_asymptotes.csv'
PARTICLES = FIRST_STAGE / 'patient_olambda_particles.csv'
OUTCOMES = DATA / 'clinical_outcomes.csv'
OUTPUT = DATA / 'bayesian_Olambda_survival' / 'lifetime_no_floor_adjusted_midpoint_v1'
DRAWS, TUNE, CHAINS, CORES = 2000, 2000, 4, 4
TARGET_ACCEPT, SEED = 0.99, 0
for path in (ESTIMATES, PARTICLES, OUTCOMES):
    if not path.is_file():
        raise FileNotFoundError(path)

In [ ]:
REVISION = '42949bfb686819b08054293d42bea6e671452a29'
repo = Path(tempfile.mkdtemp(prefix='direct-survival-')) / 'repo'
subprocess.run(['git', 'clone', '--quiet', '--single-branch', '--branch', 'olambda-lifetime-no-floor',
                'https://github.com/adamnik/CAR-T-spectral-organization.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--quiet', '--detach', REVISION], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pymc==5.28.5',
                'arviz>=0.22,<1', 'h5netcdf>=1.5'], check=True)
print('Survival model revision:', REVISION)

In [ ]:
keys = ['Dataset', 'sample_id']
est = pd.read_csv(ESTIMATES).set_index(keys, verify_integrity=True)
particles = pd.read_csv(PARTICLES).set_index(keys, verify_integrity=True)
assert est.model.eq('latent_timescale_student_t3_scale_0.35_no_asymptote_floor').all()
assert len(est) == 163 and est.index.get_level_values('Dataset').nunique() == 8
assert est.index.sort_values().equals(particles.index.sort_values())
assert list(particles) == [f'particle_{i:02d}' for i in range(1,17)]
assert (particles.gt(1) & particles.lt(float('inf'))).all().all()
# Audit eligibility with the same loader used by both fit commands in a fresh Python process.
audit = """
from survival.survival_model import load_olambda_analysis
import sys
d, *_ = load_olambda_analysis(sys.argv[1], sys.argv[2], 1000, 16, sys.argv[3], ['Treatment','Disease'])
print(d.groupby('Dataset').agg(patients=('sample_id','size'), events=('event_observed','sum')).to_string(), flush=True)
assert len(d)==163 and d.Dataset.nunique()==8, 'Clinical join lost expected patients'
"""
subprocess.run([sys.executable, '-c', audit, ESTIMATES, OUTCOMES, PARTICLES], cwd=repo, check=True)

In [ ]:
common = ['--particles-csv', str(PARTICLES), '--minimum-cells', '1000', '--n-particles', '16',
          '--draws', str(DRAWS), '--tune', str(TUNE), '--chains', str(CHAINS), '--cores', str(CORES),
          '--target-accept', str(TARGET_ACCEPT), '--random-state', str(SEED),
          '--adjustment', 'Treatment=axicel', '--adjustment', 'Disease=lymphoma']
hashes = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in (ESTIMATES, PARTICLES, OUTCOMES)}

def fit(module, name, extra=()):
    config = dict(revision=REVISION, module=module, arguments=common+list(extra), input_sha256=hashes)
    tag = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:12]
    dest = OUTPUT / name / tag
    marker = dest / 'run_manifest.json'  # Written by the model after the posterior and all tables.
    if marker.is_file():
        print('Using completed fit:', dest, flush=True)
    else:
        dest.parent.mkdir(parents=True, exist_ok=True)
        stage = Path(tempfile.mkdtemp(prefix=tag+'-running-', dir=dest.parent))
        print('Fitting', name, '| output:', dest, flush=True)
        command = [sys.executable, '-u', '-m', 'survival.'+module, ESTIMATES, OUTCOMES,
                   stage, *common, *extra]
        with subprocess.Popen(command, cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1) as proc:
            for line in proc.stdout:
                print(line, end='', flush=True)
            if proc.wait():
                raise subprocess.CalledProcessError(proc.returncode, command)
        (stage / 'notebook_config.json').write_text(json.dumps(config, indent=2))
        stage.rename(dest)
    if json.loads((dest/'notebook_config.json').read_text()) != config:
        raise ValueError(f'Run configuration mismatch: {dest}')
    print('Saved posterior:', dest/'posterior.nc')
    display(pd.read_csv(dest/'model_result.csv'))
    return dest

In [ ]:
continuous = fit('fit_olambda_survival_continuous', 'continuous')

In [ ]:
transition = fit('fit_olambda_survival_regimes', 'smooth_transition_Q10_Q90_midpoint',
                 ['--midpoint-centered', '--crossover-quantile-low', '0.10',
                  '--crossover-quantile-high', '0.90', '--transition-scale-sd', '0.10'])
manifest = json.loads((transition/'run_manifest.json').read_text())
assert manifest['hazard_activation_offset'] == 0.5
membership = pd.read_csv(transition/'patient_regime_probabilities.csv')
assert len(membership) == 163 and membership.posterior_mean_probability_high.between(0,1).all()
print('High-Oλ membership table:', transition/'patient_regime_probabilities.csv')
display(membership.head())